# Imports and Config

## Import libraries

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split

## Fix configurations

In [2]:
BATCH_SIZE = 8
EMBEDDING_DIM = 128
UNITS = 128
EPOCHS = 10

# Load Dataset

## Import data from Kaggle

In [3]:
import kagglehub

path = kagglehub.dataset_download("dhruvildave/en-fr-translation-dataset")

Using Colab cache for faster access to the 'en-fr-translation-dataset' dataset.


In [4]:
print(path)

/kaggle/input/en-fr-translation-dataset


## Load data

In [5]:
import os
file_path = os.path.join(path, "en-fr.csv")
df = pd.read_csv(file_path,nrows=100000)

In [6]:
df.head(5)

,en,fr
0,Changing Lives | Changing Society | How It Wor...,Il a transformé notre vie | Il a transformé la...
1,Site map,Plan du site
2,Feedback,Rétroaction
3,Credits,Crédits
4,Français,English


In [7]:
df.shape

(100000, 2)

# Data Preprocessing

## Drop missing records

In [8]:
df = df[['en', 'fr']].dropna()

In [9]:
df.shape

(99998, 2)

## Text Preprocessing

### Preprocessing Function

In [10]:
def preprocess_sentence(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"([?.!,¿])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-zA-ZÀ-ÿ?.!,¿]+", " ", sentence)
    sentence = sentence.strip()
    return sentence

### Apply function

In [11]:
df['en'] = df['en'].apply(preprocess_sentence)
df['fr'] = df['fr'].apply(preprocess_sentence)

### Add tokens

In [12]:
df['fr_in'] = "<start> " + df['fr']
df['fr_out'] = df['fr'] + " <end>"

In [13]:
df.head()

,en,fr,fr_in,fr_out
0,changing lives changing society how it works t...,il a transformé notre vie il a transformé la s...,<start> il a transformé notre vie il a transfo...,il a transformé notre vie il a transformé la s...
1,site map,plan du site,<start> plan du site,plan du site <end>
2,feedback,rétroaction,<start> rétroaction,rétroaction <end>
3,credits,crédits,<start> crédits,crédits <end>
4,français,english,<start> english,english <end>


## Tokenization

In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [15]:
en_tokenizer = Tokenizer(filters='',num_words=10000)
en_tokenizer.fit_on_texts(df['en'])

In [16]:
fr_tokenizer = Tokenizer(filters='',num_words=10000)
fr_tokenizer.fit_on_texts(df['fr_in'])

In [17]:
encoder_input = en_tokenizer.texts_to_sequences(df['en'])
decoder_input = fr_tokenizer.texts_to_sequences(df['fr_in'])
decoder_target = fr_tokenizer.texts_to_sequences(df['fr_out'])

In [18]:
max_len_en = 80
max_len_fr = 80

encoder_input = pad_sequences(encoder_input, maxlen=max_len_en, padding='post')
decoder_input = pad_sequences(decoder_input, maxlen=max_len_fr, padding='post')
decoder_target = pad_sequences(decoder_target, maxlen=max_len_fr, padding='post')

# Dataset Pipeline

In [19]:
df = tf.data.Dataset.from_tensor_slices((
    (encoder_input, decoder_input),
    decoder_target
))

df = df.shuffle(5000).batch(BATCH_SIZE)

# Model Architecture

## Encoder

In [20]:
class Encoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_state=True)

    def call(self, x):
        x = self.embedding(x)
        output, h, c = self.lstm(x)
        return h, c

## Decoder

In [21]:
class Decoder(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(units, return_sequences=True, return_state=True)
        self.fc = tf.keras.layers.Dense(vocab_size)

    def call(self, x, states):
        x = self.embedding(x)
        output, h, c = self.lstm(x, initial_state=states)
        output = self.fc(output)
        return output, h, c

# Training

In [22]:
encoder = Encoder(10000, EMBEDDING_DIM, UNITS)
decoder = Decoder(10000, EMBEDDING_DIM, UNITS)

In [23]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

optimizer = tf.keras.optimizers.Adam()

@tf.function
def train_step(enc_in, dec_in, dec_out):
    loss = 0

    with tf.GradientTape() as tape:
        h, c = encoder(enc_in)
        predictions, _, _ = decoder(dec_in, [h, c])

        loss = loss_fn(dec_out, predictions)

    variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))

    return loss

In [24]:
for epoch in range(EPOCHS):
    total_loss = 0
    num_batches = 0

    for (encoder_input, decoder_input), decoder_target in df:
        loss = train_step(encoder_input, decoder_input, decoder_target)
        total_loss += loss.numpy()
        num_batches += 1

    avg_loss = total_loss / num_batches
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 1.4387
Epoch 2, Loss: 1.1675
Epoch 3, Loss: 1.0659
Epoch 4, Loss: 1.0068
Epoch 5, Loss: 0.9655
Epoch 6, Loss: 0.9339
Epoch 7, Loss: 0.9084
Epoch 8, Loss: 0.8872
Epoch 9, Loss: 0.8692
Epoch 10, Loss: 0.8533


# Test

In [27]:
def encode_input(sentence):
    seq = en_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len_en, padding='post')
    return encoder(seq)

In [28]:
index_word = {v:k for k,v in fr_tokenizer.word_index.items()}

def translate(sentence):
    h, c = encode_input(sentence)

    dec_input = tf.expand_dims([fr_tokenizer.word_index['<start>']], 0)

    result = []

    for _ in range(max_len_fr):
        preds, h, c = decoder(dec_input, [h, c])

        pred_id = tf.argmax(preds[0, -1]).numpy()
        word = index_word.get(pred_id, '')

        if word == '<end>':
            break

        result.append(word)

        dec_input = tf.expand_dims([pred_id], 0)

    return ' '.join(result)

In [33]:
print(translate("I am tired"))

je suis ?                                                                             
